# 13 — M2: nt-resolution eCLIP profile conditioned on protein

**What.** A single protein-conditioned head predicts the per-nucleotide eCLIP profile for any RBP from its protein rep; does conditioning on the *right* protein beat a *wrong* one, and does it replicate across cells?

**Why.** M2 is the real multimodal contribution (profile resolution), built on gudiyi's `data/multimodal.py` sample layer and the M1+interpretability findings.

**Data (Moyon/Marsico lab).** Frozen PARNET `parnet.7m-0.0`, lab binding `binding/{pureclip,narrowpeak_intersect}`, full-223 `encode.filtered.hfds`, ESM/ProtT5/per-residue/STRING, ATtRACT domains. All numbers from committed `mmpartnet_out/*.json`; figures via the reusable `scripts/viz.py` builders, so the same notebook re-plots any teammate's same-schema results. **In-distribution** (all-223 PARNET) unless noted; leave-out PARNET is the decisive follow-up.

## Definitions

Frozen PARNET body $H$ + protein $e_g$ → per-nt target/control logits; profiles $p^t=\mathrm{softmax}(t)$; additive mixture $p=m\,p^t+(1-m)\,p^c$; count-weighted MultinomialNLL $-\frac1{\sum o_i}\sum_i o_i\log p_i$. Score = profile Pearson $r(p^t,o/\sum o)$ on held-out windows, with derangement + within-family protein-shuffle controls. Lab full-223 hfds, HepG2 & K562.

In [ ]:
import os, sys, json, pathlib
import numpy as np, matplotlib.pyplot as plt
from IPython.display import Markdown, display
_here=pathlib.Path.cwd().resolve()
REPO=next((c for c in (_here,*_here.parents) if (c/'src'/'mmpartnet').is_dir()),_here)
sys.path.insert(0,str(REPO/'scripts')); import viz
OUT=REPO/'mmpartnet_out'; FIGD=REPO/'notebooks'/'demo'/'executed'
def J(n): return json.loads((OUT/n).read_text())
from IPython.display import Image as _Img
def show(fig,name,sup=None):
    if sup: fig.suptitle(sup,fontsize=12.5,fontweight='bold',y=1.02)
    fig.savefig(str(FIGD/name),bbox_inches='tight',dpi=150); plt.close(fig)
    display(_Img(filename=str(FIGD/name)))   # embed the saved PNG inline (backend-independent)
h=J('m2_profile.json'); k=J('m2_profile_K562.json')
for nm,d in [('HepG2',h),('K562',k)]:
    for a,v in d['archs'].items(): print(f"  {nm:5} {a:7} Pearson real={v['real']:.3f} shuf={v['shuf']:.3f} gap={v['gap_der']:+.3f}")

In [ ]:
import os, sys, json, pathlib
import numpy as np, matplotlib.pyplot as plt
from IPython.display import Markdown, display
_here=pathlib.Path.cwd().resolve()
REPO=next((c for c in (_here,*_here.parents) if (c/'src'/'mmpartnet').is_dir()),_here)
sys.path.insert(0,str(REPO/'scripts')); import viz
OUT=REPO/'mmpartnet_out'; FIGD=REPO/'notebooks'/'demo'/'executed'
def J(n): return json.loads((OUT/n).read_text())
from IPython.display import Image as _Img
def show(fig,name,sup=None):
    if sup: fig.suptitle(sup,fontsize=12.5,fontweight='bold',y=1.02)
    fig.savefig(str(FIGD/name),bbox_inches='tight',dpi=150); plt.close(fig)
    display(_Img(filename=str(FIGD/name)))   # embed the saved PNG inline (backend-independent)
show(viz.fig_m2([('HepG2',J('m2_profile.json')),('K562',J('m2_profile_K562.json'))]),'nb13_m2_profile.png',
     'M2 profile conditioning: per-RBP Pearson, replicated across cells')

In [ ]:
import os, sys, json, pathlib
import numpy as np, matplotlib.pyplot as plt
from IPython.display import Markdown, display
_here=pathlib.Path.cwd().resolve()
REPO=next((c for c in (_here,*_here.parents) if (c/'src'/'mmpartnet').is_dir()),_here)
sys.path.insert(0,str(REPO/'scripts')); import viz
OUT=REPO/'mmpartnet_out'; FIGD=REPO/'notebooks'/'demo'/'executed'
def J(n): return json.loads((OUT/n).read_text())
from IPython.display import Image as _Img
def show(fig,name,sup=None):
    if sup: fig.suptitle(sup,fontsize=12.5,fontweight='bold',y=1.02)
    fig.savefig(str(FIGD/name),bbox_inches='tight',dpi=150); plt.close(fig)
    display(_Img(filename=str(FIGD/name)))   # embed the saved PNG inline (backend-independent)
h=J('m2_profile.json'); k=J('m2_profile_K562.json')
hb=max(h['archs'],key=lambda a:h['archs'][a]['gap_der'])
display(Markdown(f'''**Result.** Conditioning on the correct protein roughly **doubles** the per-nt profile Pearson (HepG2 best {hb}: {h['archs'][hb]['real']:.3f} vs {h['archs'][hb]['shuf']:.3f}, gap **{h['archs'][hb]['gap_der']:+.3f}**), replicated on K562 (gap {max(k['archs'][a]['gap_der'] for a in k['archs']):+.3f}), per-residue ≥ FiLM, mostly specific-protein (within-family gap retained). **M2 is computed.**'''))

## Conclusion

A single protein-conditioned head predicts the nt-resolution profile for any RBP; the gain is large, cross-cell robust, and specific-protein — the start of M2. Caveat: in-distribution; the leave-out PARNET swap makes it a clean held-out-RBP claim (harness unchanged).

Claude-assisted; methods per Horlacher 2023 (RBPNet) + TFBindFormer/CORAL (cross-attention).